In [516]:
import random
from fractions import Fraction
from typing import List, Tuple
import pandas as pd
from datasets import Dataset

import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

from utils.corrupt_num import *
from utils.chunk_r import *

## Evals - How to Scale:
1. number of operands
2. new operations

In [169]:
nums = [ 47, 2, 42, 9, 62, 20, 64 ]
target = 228
question = f"Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once"

In [170]:
tokenizer = AutoTokenizer.from_pretrained("zwhe99/DeepMath-1.5B")

In [171]:
conversations = [
    {"role": "system", "content": "Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>."},
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(
    conversations,
    tokenize=False,
    add_generation_prompt=True
)

In [172]:
print(prompt)

<｜begin▁of▁sentence｜>Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>.<｜User｜>Using the numbers [47, 2, 42, 9, 62, 20, 64], create an equation that equals 228. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once<｜Assistant｜><think>



In [ ]:
"Please reason and answer the following question. At the end, you should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>. Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) in the expression and each number can only be used once."

In [173]:
47 + 62 + 9 + 64 + 42/2

203.0

# Check Eval Dataset

In [177]:
cd /home/al2644/research/codebase/reasoning/perturb-r/

/home/al2644/research/codebase/reasoning/perturb-r


In [293]:
path = "./data/countdown"
dataset = load_from_disk(path)

In [296]:
dataset['test'][0]

{'nums': [32, 8, 16],
 'target': 4,
 'example_solution': '((16*8)/32)',
 'problem': 'Please reason and answer the following question. At the end, you should answer with the equation expression, for example \\boxed{(1 + 2) / 3}. Using the numbers [32, 8, 16], create an equation that equals 4. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once.',
 'level': 3,
 'task': 'countdown'}

In [404]:
df = pd.DataFrame(dataset['test'])
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')

In [362]:
dataset = load_from_disk('/share/goyal/lio/reasoning/data/countdown/sft/level3-4/')

In [366]:
root = '/share/goyal/lio/reasoning/data/countdown/sft/teacher_level3-4/'
df = pd.read_pickle(os.path.join(root, 'Qwen3-4B.pickle'))
os.listdir(root)

['Qwen3-4B.pickle']

In [379]:
idx = 10
row = df.iloc[idx]

In [382]:
def format_dataset(row, tokenizer = tokenizer):
    problem = row['problem']
    response = row['response']
    conversation = [
        {"role": "user", "content": problem},
        {"role": "assistant", "content": response}
    ]

    return tokenizer.apply_chat_template(conversation, tokenize=False)

In [383]:
result = format_dataset(row)

In [388]:
result

"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nPlease reason and answer the following question. Using the numbers [12, 89, 69], create an equation that equals 146. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. I need to use the numbers 12, 89, and 69 exactly once each, with basic operations like addition, subtraction, multiplication, or division to make an equation that equals 146. Hmm, let's start by thinking about possible combinations.\n\nFirst, maybe I should check if adding two numbers and then doing something with the third could work. Let me try adding 12 and 89. That gives 101. Then what do I do with 69? If I add 101 and 69, that's 170, which is too big. If I subtract 69 from 101, that's 32, not

In [417]:
root = "/home/al2644/research/codebase/reasoning/perturb-r/data/countdown"
dataset = load_from_disk(root)
df = pd.DataFrame(dataset['test'])

In [418]:
df

,nums,target,example_solution,problem,level,task
0,"[69, 61, 92]",84,((61-69)+92),Please reason and answer the following questio...,3,countdown
1,"[54, 30, 8]",32,((8+54)-30),Please reason and answer the following questio...,3,countdown
2,"[97, 58, 96]",57,((58-97)+96),Please reason and answer the following questio...,3,countdown
3,"[60, 15, 94]",49,((94+15)-60),Please reason and answer the following questio...,3,countdown
4,"[35, 20, 76]",131,((20+35)+76),Please reason and answer the following questio...,3,countdown
...,...,...,...,...,...,...
595,"[8, 11, 11, 3, 92, 20, 72, 37, 13, 42, 54, 11,...",379,((((((11-8)-(37+11))-11)-(((98+13)+(92*3))+72)...,Please reason and answer the following questio...,13,countdown
596,"[78, 75, 29, 27, 54, 31, 86, 43, 55, 89, 76, 2...",471,((((55+78)+86)*(54-29))-((43-31)+((((89+75)+76...,Please reason and answer the following questio...,13,countdown
597,"[92, 72, 88, 20, 75, 40, 67, 46, 29, 44, 88, 3...",188,((29-92)+((((((88-20)+75)-((38-44)-40))+67)+46...,Please reason and answer the following questio...,13,countdown
598,"[42, 81, 86, 2, 46, 48, 56, 37, 92, 93, 90, 4,...",977,((((37+46)-(2+42))-(((56-86)*30)-48))-((((93+9...,Please reason and answer the following questio...,13,countdown


In [415]:
idx = 0
example = df.iloc[idx]
numbers, target = example['nums'], example['target']
problem = f"Please reason and answer the following question. Using the numbers {numbers}, create an equation that equals {target}. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>."
conversation = [{'role': 'user', 'content': problem}]
prompt_str = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)

In [416]:
print(prompt_str)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [32, 8, 16], create an equation that equals 4. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant



# Reasoning Model Eval

In [614]:
root = "./results/countdown/benchmark/"
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
for fname in os.listdir(root):
    if 'correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['correct'].mean()}")

# model specific
df = pd.read_pickle(os.path.join(root, 'Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle'))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.24214285714285713
Qwen2.5-3B-distill-countdown-level3-4-3epoch.pickle
Accuracy: 0.24714285714285714
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.35


In [612]:
def groupby_level (df):
    stats = df.groupby('level')['correct'].mean()
    return stats

# Corrupted Numbers

In [704]:
root = "./results/countdown/corrupt_numbers/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    if 'still_correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['still_correct'].mean()}")

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.5202702702702703
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.5934684684684685


In [694]:
wrong_answer = df[(df['still_correct']==0.) & (df['if_answer'])]
correct_answer = df[df['still_correct']==1.]

In [695]:
wrong_answer[['level', 'type']].value_counts()

level  type         
5      post_</think>    80
6      post_</think>    21
5      pre_</think>     17
6      pre_</think>      5
Name: count, dtype: int64

In [705]:
print(df[df['type'] == 'pre_<think>']['prompt'].iloc[10])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [35, 48, 94, 46, 4], create an equation that equals 140. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.

This is user's thought process for this problem: <think>
Okay, let's see. I need to use the numbers 35, 48, 94, 46, and 4 exactly once each, with basic arithmetic operations to make 140. Hmm. Let me start by thinking about possible combinations. 

First, maybe I should check if adding some numbers and subtracting others could work. Let's list the numbers again: 35, 48, 94, 46, 4. The target is 140. Let me try some combinations.

What if I subtract two numbers? For example, 94 minus something? Let's see. If I take 94 and subtract 46, that gives 48. Then

In [706]:
idx = 21
example = correct_answer.iloc[idx]
prompt, response = example['prompt'], example['post_corruption_response']

In [707]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [43, 77, 79, 50, 96], create an equation that equals 259. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to use the numbers 43, 77, 79, 50, and 96 exactly once each with basic operations to make 259. Hmm. Let me start by thinking about possible combinations.

First, maybe I should check if adding some numbers and subtracting others could work. Let me try adding two numbers and subtracting the others. For example, 96 + 79 is 175. Then 77 + 50 is 127. Then 175 + 127 is 302. That's too high. Maybe subtract something? If I subtract 43, 302 - 43 is 259. Wait, that
seems promising. Let me check:




In [708]:
print(response)

the original numbers? Wait, no, the original numbers are 43, 77, 79, 50, 96. So maybe that's not the right approach. Wait, maybe I need to use all numbers once in the equation. So the idea would be (96 + 79) + (77 + 50) - 43 = 259. Let me compute that:

96 + 79 = 175. 77 + 50 = 127. 175 + 127 = 302. 302 - 43 = 259. Yes! That works. But wait, how do I arrange the operations? Let me check that each number is used exactly once. 96, 79, 77, 50, 43. Yes, all are used once. So the equation would be (96 + 79) + (77 + 50) - 43 = 259. Let me verify again:

96 + 79 is 175. 77 + 50 is 127. 175 + 127 is 302. 302 - 43 is 259. Correct. So that's the answer. But maybe I need to check if there's another way, but this seems to work. Let me write it properly in the required format. The expression would be (96 + 79) + (77 + 50) - 43. So that's the answer.
</think>

<answer> (96 + 79) + (77 + 50) - 43 </answer>

